In [2]:
import argparse
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans as SKKMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
OUT = "outputs_segmentation"
os.makedirs(OUT, exist_ok=True)

# K-mean from scratch (Numpy) with k-mean++ initialization

In [23]:

class Kmeans_scrath:
    def __init__(self,k,max_iters=300,n_init=10,tol=1e-6,seed=24):# tol=حدّ التغيّر الذي عنده تعتبر الخوارزمية أنها تقاربت وتتوقف. seed=قيمة لتثبيت العشوائية، بحيث تكون النتائج قابلة للتكرار.
       self.k,self.max_iters,self.n_init=k,max_iters,n_init
       self.tol, self.seed=tol,seed
       self.centriods,self.labels_,self.inertia_=None,None,None
     
    def _init_centroids(self,x,rng):
        #k_mmeans++ pick far_apart starting centroids
        n=x.shape[0]
        c=[x[rng.integers(n)]] # sselect random postion r=from to n-1 to start
        for _ in range(1,self.k):
            d2=np.min(((x[:,None,:]-np.array(c)[None,:,:])**2).sum(2),axis=1) # for all point in x calculate squerd deciitance betweem this and all centroid was selected  and stored the litter one in d2
            probs=d2/d2.sum() #يحوّل هذه المسافات إلى احتمالات للاختيار. النقاط الأبعد عن أقرب مركز تحصل على احتمال أكبر.
            c.append(x[rng.choice(n,p=probs)])#  ثم يضيفها الي قايمه المراكز  ي  probs ختار نقطة جديدة عشوائيًا وفق الاحتمالات          
        return np.array(c)  
    def _assign(self,x,c):#هذه الدالة تُسنِد كل نقطة بيانات إلى أقرب مركز، وتحسب مجموع مربعات المسافات إلى المراكز المختارة.
        d2=((x[:,None,:]-c[None,:,:])**2).sum(2)
        return d2.argmin(1),d2.min(1).sum()   #إذًا الدالة تعيد زوجًا: تسميات المجموعات ومجموع مربعات المسافات إلى أقرب المراكز.
    
    def fit_once(self,x,rng):#هذه الدالة تنفّذ تشغيلًا واحدًا من خوارزمية K-Means: تبدأ بمراكز أولية، ثم تكرّر إسناد النقاط وتحديث المراكز.
        
        c=self._init_centroids(x,rng)# تختار المراكز الابتدائية
        for _ in  range(self.max_iters):
            labels,inertia=self._assign(x,c)#تُسنِد كل نقطة إلى أقرب مركز، وتحسب قيمة inertia
            new_c=c.copy()
            for j in range (self.k):
                pts=x[labels==j]
                if len(pts):#ذا كانت المجموعة تحتوي نقاطًا، يجعل مركزها الجديد هو متوسط تلك النقاط عبر أبعاد البيانات
                    new_c[j]=pts.mean(0)
                else:    
                    new_c[j]=x[rng.integers(len(x))]
            shift=np.abs(new_c-c).max()#يحسب أكبر تغيّر مطلق بين المراكز القديمة والجديدة، ثم يعتمد المراكز الجديدة.
            c=new_c
            if shift < self.tol:
               break     
        labels,inertia=self._assign(x,c)
        return c,labels,inertia      
    def fit(self,x):
        rng=np.random.default_rng(self.seed) # لتكون النتائج قابلة للتكرار عند استخدام الإعدادات والبيانات نفسهاseed،#ينشئ مولّد أرقام عشوائية باستخدام    
        best=None
        for _ in range(self.n_init):
            res=self.fit_once(x,rng)
            if best is None or res[2]<best[2]:#   inertia يحتفظ بنتيجة التشغيل التي حققت أقل أي أقل مجموع لمربعات المسافات إلى المراكز
                best=res
        self.centroids,self.labels_,self.inertia_=best # يخزّن أفضل المراكز والتسميات وقيمة  اخل الكائن، ثم يعيد الكائن نفسه. لذلك يمكن كتابة
     
        
        return self 
    def predict(self,x):
        return self.assign(x,self.centroids)[0]        

 Load data

In [24]:
def load_data(demo=False, path="Mall_Customers.csv"):
    if demo or not os.path.exists(path):
        print(">> Using synthetic demo data (Mall_Customers.csv not found)")
        rng = np.random.default_rng(0)
        centers = [(25, 25, 78), (45, 25, 20), (32, 85, 82), (42, 88, 17), (45, 55, 50)] #ينشئ مولّدًا عشوائيًا ثابتًا للتكرار، ويحدد خمسة مراكز تقريبية. كل مركز يمثل قيمًا تقريبية لـ العمر، والدخل السنوي، ودرجة الإنفاق
        rows = []
        for a, inc, sp in centers:
            rows.append(np.c_[rng.normal(a, 5, 40), rng.normal(inc, 6, 40), rng.normal(sp, 8, 40)])# كل مركز، يولّد 40 صفًا حول قيمه باستخدام توزيع طبيعي. الأرقام 5 و6 و8 تحدد مقدار التشتت حول كل قيمة.
        X = np.vstack(rows) #يجمع الصفوف الخمسة في مصفوفة واحدة حجمها 200 × 3: مئتا عميل وثلاث ميزات
        df = pd.DataFrame(X, columns=["Age", "Annual Income (k$)", "Spending Score (1-100)"])
        df["Gender"] = rng.choice(["Male", "Female"], len(df))#يضع القيم في جدول بأسماء أعمدة واضحة، ثم يضيف عمودًا للجنس عشوائيًا ويعيد الجدول. هذا العمود لن يدخل في التدريب أدناه.
        return df
    return pd.read_csv(path)


def main(demo):
    df = load_data(demo)
    print(df.head(), "\n")
    print(df.describe(), "\n")
    print("Missing values:\n", df.isnull().sum(), "\n")

    features = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
    X_raw = df[features].values
    X = StandardScaler().fit_transform(X_raw)  # K-means is distance-based -> scale!
    
    
     # ------------------ choose K: Elbow + Silhouette ------------------
    ks = range(2, 11)#ينشئ القيم من 2 إلى 10 شاملًا، ويجهّز قائمتين لتخزين النتائج
    inertias, sils = [], []
    for k in ks:
        km = Kmeans_scrath(k).fit(X)
        inertias.append(km.inertia_)
        sils.append(silhouette_score(X, km.labels_))
        """ينشئ نموذج K-Means بعدد مجموعات k ويدرّبه على X.
        يخزّن inertia_: مجموع مربعات المسافات بين النقاط ومراكز مجموعاتها.
        يحسب درجة Silhouette باستخدام النقاط وتسميات مجموعاتها.
        درجة Silhouette الأعلى تعني عادةً أن المجموعات أكثر تماسكًا وأوضح انفصالًا عن بعضها.
        """
    best_k = list(ks)[int(np.argmax(sils))] # يجد موضع أعلى درجة Silhouette، ثم يستخدم هذا الموضع لاستخراج قيمة K المقابلة له. وبما أن ks يبدأ من 2، فمثلًا أعلى درجة عند الموضع الأول تعني أن best_k = 2
    
    print(f"Silhouette scores: { {k: round(s, 3) for k, s in zip(ks, sils)} }")
    print(f"Best K by silhouette = {best_k}\n")

    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(list(ks), inertias, "o-"); ax[0].set_title("Elbow method")
    ax[0].set_xlabel("K"); ax[0].set_ylabel("Inertia (cost J)")
    ax[1].plot(list(ks), sils, "o-", color="green"); ax[1].set_title("Silhouette score")
    ax[1].set_xlabel("K"); ax[1].set_ylabel("Score")
    plt.tight_layout(); plt.savefig(f"{OUT}/choose_k.png", dpi=150); plt.close()
    # ------------------ final model + compare with sklearn ------------------
    K = best_k  # you can override manually after looking at the elbow plot
    km =Kmeans_scrath(K, n_init=10).fit(X)
    sk = SKKMeans(n_clusters=K, n_init=10, random_state=42).fit(X)
    print(f"Scratch inertia: {km.inertia_:.2f} | sklearn inertia: {sk.inertia_:.2f}")
    print(f"Agreement (Adjusted Rand Index): {adjusted_rand_score(km.labels_, sk.labels_):.3f}\n")

    df["Cluster"] = km.labels_
    
    # ------------------ profile each cluster ------------------
    profile = df.groupby("Cluster")[features].mean().round(1)
    profile["Size"] = df["Cluster"].value_counts().sort_index()
    print("Cluster profiles:\n", profile, "\n")
    profile.to_csv(f"{OUT}/cluster_profiles.csv")
    
    # Simple automatic naming based on income/spending (relative to the data mean)
    inc_m, sp_m = df[features[1]].mean(), df[features[2]].mean()
    names = {}
    for c, r in profile.iterrows():
        hi_inc, hi_sp = r[features[1]] > inc_m, r[features[2]] > sp_m
        names[c] = {(True, True): "High income - High spending (VIP)",
                    (True, False): "High income - Low spending (Careful)",
                    (False, True): "Low income - High spending (Impulsive)",
                    (False, False): "Low income - Low spending (Budget)"}[(hi_inc, hi_sp)]
    df["Segment"] = df["Cluster"].map(names)
    print(df["Segment"].value_counts(), "\n")
    
    # ------------------ visualizations ------------------
    fig = plt.figure(figsize=(12, 5))
    ax1 = fig.add_subplot(1, 2, 1)
    sc = ax1.scatter(df[features[1]], df[features[2]], c=df["Cluster"], cmap="tab10", s=35)
    cents = StandardScaler().fit(X_raw).inverse_transform(km.centroids)
    ax1.scatter(cents[:, 1], cents[:, 2], c="black", marker="X", s=200, label="centroid")
    ax1.set_xlabel(features[1]); ax1.set_ylabel(features[2])
    ax1.set_title("Income vs Spending"); ax1.legend()

    pca = PCA(n_components=2).fit(X)
    Z = pca.transform(X)
    ax2 = fig.add_subplot(1, 2, 2)
    ax2.scatter(Z[:, 0], Z[:, 1], c=df["Cluster"], cmap="tab10", s=35)
    ax2.set_title(f"PCA view (explained var = {pca.explained_variance_ratio_.sum():.0%})")
    plt.tight_layout(); plt.savefig(f"{OUT}/clusters.png", dpi=150); plt.close()

    df.to_csv(f"{OUT}/customers_with_segments.csv", index=False)
    print(f"Done. Results saved in ./{OUT}/")
if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--demo", action="store_true")
    args, _ = ap.parse_known_args()
    main(args.demo)

   CustomerID  Gender  Age  Annual Income (k$)  Spending Score (1-100)
0           1    Male   19                  15                      39
1           2    Male   21                  15                      81
2           3  Female   20                  16                       6
3           4  Female   23                  16                      77
4           5  Female   31                  17                      40 

       CustomerID         Age  Annual Income (k$)  Spending Score (1-100)
count  200.000000  200.000000          200.000000              200.000000
mean   100.500000   38.850000           60.560000               50.200000
std     57.879185   13.969007           26.264721               25.823522
min      1.000000   18.000000           15.000000                1.000000
25%     50.750000   28.750000           41.500000               34.750000
50%    100.500000   36.000000           61.500000               50.000000
75%    150.250000   49.000000           78.000000     

choose K: Elbow + Silhouette